<a href="https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/NB03_taxon_directed_homologs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB03 — Taxon-directed BLASTP

This notebook is a course adaptation of the original
`BLASTp_taxon_directed_v4_scratch` workflow.

Both required input files are stored in the data folder associated with this notebook:

- `translated_query_protein.faa`
- `taxon_targets.tsv`

The main BLASTP workflow remains essentially the same as the original V4 notebook.


In [12]:
!fusermount -u /content/drive 2>/dev/null || true
!rm -rf /content/drive 2>/dev/null || true
!mkdir -p /content/drive

In [13]:
# Cell 0: Mount Google Drive and locate the ACTUAL NB03 data folder

from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

NOTEBOOK_FOLDER_NAME = "NB03_taxon_directed_homologs"
QUERY_FILENAME = "translated_query_protein.faa"
TAXON_FILENAME = "taxon_targets.tsv"

# First check the two normal course locations.
candidate_data_dirs = [
    f"/content/drive/MyDrive/BIOINFO4-5203-F26/Data/{NOTEBOOK_FOLDER_NAME}",
    f"/content/drive/MyDrive/Teaching/BIOINFO4-5203-F26/Data/{NOTEBOOK_FOLDER_NAME}",
]

def has_both_inputs(folder):
    return (
        os.path.isdir(folder)
        and os.path.isfile(os.path.join(folder, QUERY_FILENAME))
        and os.path.isfile(os.path.join(folder, TAXON_FILENAME))
    )

matches = [folder for folder in candidate_data_dirs if has_both_inputs(folder)]

# If neither standard location contains BOTH files, search MyDrive for the
# NB03 folder itself. This handles alternate course-folder locations.
if not matches:
    print("NB03 inputs were not found in the two standard locations.")
    print("Searching MyDrive for the NB03 data folder...")

    search_root = "/content/drive/MyDrive"

    for root, dirs, files in os.walk(search_root):
        # Only accept a directory with the exact NB03 folder name
        # AND both required input files.
        if os.path.basename(root) == NOTEBOOK_FOLDER_NAME:
            if QUERY_FILENAME in files and TAXON_FILENAME in files:
                matches.append(root)

if not matches:
    raise FileNotFoundError(
        "Could not find a folder named "
        f"'{NOTEBOOK_FOLDER_NAME}' containing BOTH required files:\n"
        f"  {QUERY_FILENAME}\n"
        f"  {TAXON_FILENAME}\n\n"
        "Check that Google Drive is mounted and that both files are in the "
        "same NB03 data folder."
    )

# If more than one valid copy exists, show them and use the first match.
if len(matches) > 1:
    print("More than one valid NB03 data folder was found:")
    for i, folder in enumerate(matches, start=1):
        print(f"  {i}. {folder}")
    print("\nUsing the first valid folder.")

NOTEBOOK_DATA_DIR = matches[0]

# Derive the corresponding course root from .../Data/NB03_taxon_directed_homologs
DATA_PARENT = os.path.dirname(NOTEBOOK_DATA_DIR)   # .../Data
PROJECT_ROOT = os.path.dirname(DATA_PARENT)        # course root
NOTEBOOK_OUTPUT_DIR = os.path.join(
    PROJECT_ROOT, "Outputs", NOTEBOOK_FOLDER_NAME
)

os.makedirs(NOTEBOOK_OUTPUT_DIR, exist_ok=True)

print("\nFOUND NB03 INPUT DIRECTORY:")
print(" ", NOTEBOOK_DATA_DIR)

print("\nFiles in this directory:")
for name in sorted(os.listdir(NOTEBOOK_DATA_DIR)):
    print(" ", name)

print("\nOutput directory:")
print(" ", NOTEBOOK_OUTPUT_DIR)


Mounted at /content/drive

FOUND NB03 INPUT DIRECTORY:
  /content/drive/MyDrive/Teaching/BIOINFO4-5203-F26/Data/NB03_taxon_directed_homologs

Files in this directory:
  README_INPUTS.txt
  taxon_targets.tsv
  translated_query_protein.faa

Output directory:
  /content/drive/MyDrive/Teaching/BIOINFO4-5203-F26/Outputs/NB03_taxon_directed_homologs


##Cell 1:Install dependencies (BLAST+, Biopython, pandas)

In [14]:
!apt-get update -qq
!apt-get install -y ncbi-blast+ -qq

!pip install biopython pandas -q
!apt-get install -y ncbi-blast+ -qq
!pip install biopython pandas requests -q

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


##Cell 2:Configuration: paths, NCBI email, input files

In [15]:
# Cell 2: Configuration (email, input files, output files)

import os
import pandas as pd
from Bio import Entrez, SeqIO

# REQUIRED by NCBI — use your real e-mail
Entrez.email = "rob.burnap@okstate.edu"

# ===== INPUT FILES =====
# Both were located automatically in the NB03-associated data directory.
QUERY_FASTA = os.path.join(NOTEBOOK_DATA_DIR, QUERY_FILENAME)
TAXID_LIST  = os.path.join(NOTEBOOK_DATA_DIR, TAXON_FILENAME)

# ===== OUTPUT FILES =====
SUMMARY_CSV = os.path.join(NOTEBOOK_OUTPUT_DIR, "top_hits_by_taxid.csv")
MSA_FASTA   = os.path.join(NOTEBOOK_OUTPUT_DIR, "top_hits_by_taxid.faa")

print("Query FASTA :", QUERY_FASTA)
print("TaxID List  :", TAXID_LIST)
print("Summary CSV :", SUMMARY_CSV)
print("Output FASTA:", MSA_FASTA)

print("\nInput file check:")
print("Query FASTA found:", os.path.isfile(QUERY_FASTA))
print("Taxon table found:", os.path.isfile(TAXID_LIST))

if not os.path.isfile(QUERY_FASTA) or not os.path.isfile(TAXID_LIST):
    raise FileNotFoundError(
        "The NB03 folder was located, but one of the two required inputs "
        "is no longer available."
    )


Query FASTA : /content/drive/MyDrive/Teaching/BIOINFO4-5203-F26/Data/NB03_taxon_directed_homologs/translated_query_protein.faa
TaxID List  : /content/drive/MyDrive/Teaching/BIOINFO4-5203-F26/Data/NB03_taxon_directed_homologs/taxon_targets.tsv
Summary CSV : /content/drive/MyDrive/Teaching/BIOINFO4-5203-F26/Outputs/NB03_taxon_directed_homologs/top_hits_by_taxid.csv
Output FASTA: /content/drive/MyDrive/Teaching/BIOINFO4-5203-F26/Outputs/NB03_taxon_directed_homologs/top_hits_by_taxid.faa

Input file check:
Query FASTA found: True
Taxon table found: True


#Cell 3: Set BLAST Paremters

In [16]:
#Set BLAST parameters (tune these if needed)

# BLAST parameters
BLAST_PROGRAM = "blastp"
BLAST_DB = "nr"            # NCBI 'nr' via remote BLAST
EVALUE = 1e-3
HITLIST_SIZE = 10          # fetch a small list then take the best *usable* hit
MAX_HSPS = 1

# Courtesy delay between NCBI requests (seconds)
NCBI_SLEEP = 2

print("BLAST_PROGRAM :", BLAST_PROGRAM)
print("BLAST_DB      :", BLAST_DB)
print("EVALUE        :", EVALUE)
print("HITLIST_SIZE  :", HITLIST_SIZE)
print("MAX_HSPS      :", MAX_HSPS)
print("NCBI_SLEEP    :", NCBI_SLEEP)


BLAST_PROGRAM : blastp
BLAST_DB      : nr
EVALUE        : 0.001
HITLIST_SIZE  : 10
MAX_HSPS      : 1
NCBI_SLEEP    : 2


# Cell 3: Load TaxID table (Header <tab> TaxID)


In [17]:

import csv

def load_taxids(taxid_file):
    labels, taxids = [], []

    with open(taxid_file, 'r') as fh:
        reader = csv.reader(fh, delimiter='\t')
        for row in reader:
            if len(row) < 2:
                continue

            label = row[0].strip()
            taxid = row[1].strip()

            # Skip header line
            if label.lower() == "header" and taxid.lower() == "taxid":
                continue

            if not taxid.isdigit():
                print("Skipping bad line:", row)
                continue

            labels.append(label)
            taxids.append(taxid)

    return labels, taxids

labels, taxids = load_taxids(TAXID_LIST)

print(f"Loaded {len(labels)} taxonomic labels/taxids.")
print(list(zip(labels, taxids))[:10])

Loaded 37 taxonomic labels/taxids.
[('Arthrospira platensis NIES-39', '696747'), ('Nostoc sp. PCC 7120', '103690'), ('Chroococcidiopsis thermalis PCC 7203', '251229'), ('Synechococcus sp. PCC 7002', '32049'), ('Cyanothece sp. PCC 7424', '65393'), ('Synechocystis sp. GT-S, PCC 6803', '1148'), ('Synechococcus sp. PCC 7942', '1140'), ('Halothece sp. PCC 7418', '65093'), ('Prochlorococcus marinus MIT 9313', '74547'), ('Prochlorococcus marinus pastoris CCMP 1986', '142479')]


#Cell 4 — BLAST + Entrez helpers

In [18]:
# Cell 4: BLAST + Entrez helper functions (upgraded for iTOL metadata)

import time, re
import requests
from Bio import Entrez, SeqIO
from Bio.Blast import NCBIWWW, NCBIXML

def run_taxid_blastp(query_seq, taxid, hitlist_size=10, expect=1e-5):
    """
    Remote blastp against nr restricted to a TaxID.
    Returns (best_alignment, stats_dict) or (None, None)
    """
    entrez_query = f"txid{taxid}[ORGN]"
    handle = NCBIWWW.qblast(
        program="blastp",
        database="nr",
        sequence=query_seq,
        entrez_query=entrez_query,
        hitlist_size=hitlist_size,
        expect=expect,
        format_type="XML"
    )
    blast_record = NCBIXML.read(handle)
    handle.close()

    if not blast_record.alignments:
        return None, None

    align = blast_record.alignments[0]
    hsp   = align.hsps[0]

    stats = {
        "bitscore": hsp.bits,
        "evalue": hsp.expect,
        "identity": hsp.identities,
        "align_length": hsp.align_length,
        "pident": 100*hsp.identities/max(1, hsp.align_length),
        "title": align.title,
    }
    return align, stats

def fetch_full_protein_fasta(accession):
    handle = Entrez.efetch(db="protein", id=accession, rettype="fasta", retmode="text")
    records = list(SeqIO.parse(handle, "fasta"))
    handle.close()
    return records[0] if records else None

def fetch_gb_record(accession):
    """
    GenBank record is best for organism name and db_xref (taxon, UniProt).
    """
    handle = Entrez.efetch(db="protein", id=accession, rettype="gb", retmode="text")
    rec = SeqIO.read(handle, "genbank")
    handle.close()
    return rec

def parse_taxid_from_gb(gb_record):
    """
    Extract taxid from source feature db_xref taxon:#### if present.
    """
    for feat in gb_record.features:
        if feat.type == "source":
            for x in feat.qualifiers.get("db_xref", []):
                m = re.search(r"taxon:(\d+)", x)
                if m:
                    return m.group(1)
    return ""

def parse_uniprot_from_gb(gb_record):
    """
    Try to extract UniProt from CDS db_xref lines in the GenBank record.
    Returns a ';'-joined string (possibly empty).
    """
    hits = set()
    for feat in gb_record.features:
        if feat.type != "CDS":
            continue
        for x in feat.qualifiers.get("db_xref", []):
            m = re.search(r"UniProtKB(?:/[^:]+)?:([A-Z0-9]{6,10})", x)
            if m:
                hits.add(m.group(1))
    return ";".join(sorted(hits))

def uniprot_map_refseq_to_uniprot(refseq_id, timeout=20):
    """
    Fallback: map RefSeq protein (WP_/YP_/NP_) to UniProt using UniProt ID mapping API.
    Returns UniProt accession or "".
    """
    try:
        submit = requests.post(
            "https://rest.uniprot.org/idmapping/run",
            data={"from": "RefSeq_Protein", "to": "UniProtKB", "ids": refseq_id},
            timeout=timeout
        )
        submit.raise_for_status()
        job = submit.json()["jobId"]

        # poll
        for _ in range(30):
            status = requests.get(f"https://rest.uniprot.org/idmapping/status/{job}", timeout=timeout)
            status.raise_for_status()
            js = status.json()
            if js.get("jobStatus") in (None, "FINISHED"):
                break
            time.sleep(1)

        res = requests.get(
            f"https://rest.uniprot.org/idmapping/uniprotkb/results/{job}",
            params={"format": "json"},
            timeout=timeout
        )
        res.raise_for_status()
        data = res.json()
        results = data.get("results", [])
        if results:
            return results[0]["to"]["primaryAccession"]
    except Exception:
        return ""
    return ""

#Cell 5: Main loop: BLAST each TaxID, fetch sequence

In [ ]:
import os, time, random, csv
import pandas as pd
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio import Entrez
from Bio.Blast import NCBIWWW, NCBIXML

# ---------- REQUIRED ----------
Entrez.email = "rob.burnap@okstate.edu"

# ---------- Tunables ----------
BLAST_PROGRAM = "blastp"
BLAST_DB      = "nr"
EVALUE        = 1e-5
HITLIST_SIZE  = 10
MAX_HSPS      = 1

# Courtesy timing (baseline); backoff happens only on errors
BASE_SLEEP_SEC = 0.6           # normal pause between taxa
JITTER_SEC     = 0.4           # add random jitter to reduce burstiness
MAX_RETRIES    = 4             # retries for transient connection issues
BACKOFF_START  = 2.0           # seconds (2, 4, 8, 16...)

# Optional: if qblast returns alignments that are "predicted"/low-info, you can skip by keywords.
SKIP_TITLE_KEYWORDS = ["hypothetical", "uncharacterized"]  # set [] to disable

# ---------- Files ----------
SUMMARY_CSV = SUMMARY_CSV
MSA_FASTA   = MSA_FASTA

ITOL_LABELS = f"{NOTEBOOK_OUTPUT_DIR}/iTOL_LABELS.tsv"
ITOL_POPUP  = f"{NOTEBOOK_OUTPUT_DIR}/iTOL_POPUP.tsv"

# State/Resume files
STATE_DIR   = f"{NOTEBOOK_OUTPUT_DIR}/_state"
os.makedirs(STATE_DIR, exist_ok=True)
DONE_TSV    = f"{STATE_DIR}/done_taxa.tsv"     # records completed taxa
ERR_TSV     = f"{STATE_DIR}/errors_taxa.tsv"   # records errors

# ---------- Helpers ----------
def polite_sleep(mult=1.0):
    time.sleep(mult * (BASE_SLEEP_SEC + random.random() * JITTER_SEC))

def retry_call(fn, *, what="call"):
    """
    Retry wrapper with exponential backoff + jitter.
    """
    last_err = None
    delay = BACKOFF_START
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fn()
        except Exception as e:
            last_err = e
            # backoff
            time.sleep(delay + random.random() * 0.5)
            delay *= 2
    raise last_err

def load_done_set(done_tsv_path):
    done = set()
    if os.path.exists(done_tsv_path):
        with open(done_tsv_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                # taxid \t label
                parts = line.split("\t")
                if parts:
                    done.add(parts[0])
    return done

def append_line(path, fields):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a") as f:
        f.write("\t".join(str(x) for x in fields) + "\n")

def safe_pct(num, den):
    den = max(1, int(den))
    return 100.0 * float(num) / float(den)

def make_display_label(label, acc, taxid_requested):
    # Keep iTOL leaf key stable: accession only.
    # Keep label text simple & safe.
    return f"{label} | {acc} | txid:{taxid_requested}"

def make_popup_text(label, taxid_requested, acc, stats, title):
    lines = [
        f"Group label: {label}",
        f"Requested TaxID: {taxid_requested}",
        f"NCBI accession: {acc}",
        f"BLAST title: {title}",
        "",
        f"BLAST bitscore: {stats.get('bitscore','')}",
        f"BLAST evalue: {stats.get('evalue','')}",
        f"BLAST %ident: {stats.get('pident',''):.2f}" if isinstance(stats.get("pident", None), (int,float)) else f"BLAST %ident: {stats.get('pident','')}",
        f"BLAST aln length: {stats.get('align_length','')}",
    ]
    return "\n".join(lines)

def run_taxid_blastp_xml(query_seq, taxid, hitlist_size=10, expect=1e-5):
    """
    Remote blastp against BLAST_DB restricted to a TaxID.
    Returns (blast_record) parsed from XML.
    """
    entrez_query = f"txid{taxid}[ORGN]"
    handle = NCBIWWW.qblast(
        program=BLAST_PROGRAM,
        database=BLAST_DB,
        sequence=query_seq,
        entrez_query=entrez_query,
        hitlist_size=hitlist_size,
        expect=expect,
        format_type="XML"
    )
    rec = NCBIXML.read(handle)
    handle.close()
    return rec

def pick_best_alignment(blast_record):
    """
    Choose first alignment/hsp; optionally skip low-info titles by keywords.
    """
    if not blast_record.alignments:
        return None, None

    for align in blast_record.alignments:
        if not align.hsps:
            continue
        title = (align.title or "").lower()
        if SKIP_TITLE_KEYWORDS:
            if any(k in title for k in SKIP_TITLE_KEYWORDS):
                continue
        hsp = align.hsps[0]
        stats = {
            "bitscore": hsp.bits,
            "evalue": hsp.expect,
            "identity": hsp.identities,
            "align_length": hsp.align_length,
            "pident": safe_pct(hsp.identities, hsp.align_length),
            "title": align.title,
        }
        return align, stats

    # If everything got skipped, fall back to the first alignment
    align = blast_record.alignments[0]
    hsp = align.hsps[0]
    stats = {
        "bitscore": hsp.bits,
        "evalue": hsp.expect,
        "identity": hsp.identities,
        "align_length": hsp.align_length,
        "pident": safe_pct(hsp.identities, hsp.align_length),
        "title": align.title,
    }
    return align, stats

def efetch_fasta_by_accession(acc):
    """
    Fetch protein FASTA from NCBI. Uses Entrez (more reliable than parsing BLAST output).
    """
    h = Entrez.efetch(db="protein", id=acc, rettype="fasta", retmode="text")
    recs = list(SeqIO.parse(h, "fasta"))
    h.close()
    return recs[0] if recs else None

def ensure_itol_headers(path, dataset_type):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return
    with open(path, "w") as f:
        if dataset_type == "LABELS":
            f.write("LABELS\nSEPARATOR TAB\nDATA\n")
        elif dataset_type == "POPUP":
            f.write("POPUP_INFO\nSEPARATOR TAB\nDATA\n")

def append_itol_label(acc, display_label):
    ensure_itol_headers(ITOL_LABELS, "LABELS")
    with open(ITOL_LABELS, "a") as f:
        f.write(f"{acc}\t{display_label}\n")

def append_itol_popup(acc, popup_text):
    ensure_itol_headers(ITOL_POPUP, "POPUP")
    safe = popup_text.replace("\t", " ").replace("\n", "\\n")
    with open(ITOL_POPUP, "a") as f:
        f.write(f"{acc}\t{safe}\n")

def append_fasta_record(path, rec: SeqRecord):
    # append mode for FASTA
    with open(path, "a") as f:
        SeqIO.write([rec], f, "fasta")

def append_summary_row(path, rowdict):
    df = pd.DataFrame([rowdict])
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        df.to_csv(path, index=False)
    else:
        df.to_csv(path, mode="a", header=False, index=False)

# ---------- Load query ----------
query_record = list(SeqIO.parse(QUERY_FASTA, "fasta"))[0]
query_seq = str(query_record.seq)

# ---------- Resume state ----------
done_taxids = load_done_set(DONE_TSV)
print(f"Already completed taxa: {len(done_taxids)}")

# ---------- Main loop ----------
for label, taxid in zip(labels, taxids):
    if taxid in done_taxids:
        # already done in a prior run
        continue

    print(f"TaxID {taxid} ({label}) ... ", end="")

    # 1) Remote BLAST (retry)
    try:
        blast_record = retry_call(
            lambda: run_taxid_blastp_xml(query_seq, taxid, hitlist_size=HITLIST_SIZE, expect=EVALUE),
            what=f"qblast txid{taxid}"
        )
    except Exception as e:
        print(f"BLAST error: {type(e).__name__}: {e}")
        append_line(ERR_TSV, [taxid, label, "blast_error", type(e).__name__, str(e)])
        polite_sleep(mult=2.0)
        continue

    align, stats = pick_best_alignment(blast_record)
    if align is None:
        print("no hits")
        row = {"label": label, "taxid": taxid, "status": "no_hits"}
        append_summary_row(SUMMARY_CSV, row)
        append_line(DONE_TSV, [taxid, label])
        done_taxids.add(taxid)
        polite_sleep()
        continue

    acc = align.accession
    print("hit:", acc)

    # 2) Fetch FASTA for chosen accession (retry)
    try:
        full = retry_call(lambda: efetch_fasta_by_accession(acc), what=f"efetch fasta {acc}")
    except Exception as e:
        print(f"FASTA fetch error: {type(e).__name__}: {e}")
        row = {"label": label, "taxid": taxid, "ncbi_accession": acc, "status": "fetch_fasta_error", "error": str(e)}
        append_summary_row(SUMMARY_CSV, row)
        append_line(ERR_TSV, [taxid, label, "fetch_fasta_error", acc, type(e).__name__, str(e)])
        polite_sleep(mult=2.0)
        continue

    if full is None:
        print("FASTA missing")
        row = {"label": label, "taxid": taxid, "ncbi_accession": acc, "status": "fetch_fasta_empty"}
        append_summary_row(SUMMARY_CSV, row)
        append_line(ERR_TSV, [taxid, label, "fetch_fasta_empty", acc])
        polite_sleep()
        continue

    # 3) Write FASTA incrementally (accession stays the ID for tree/iTOL)
    # Keep description minimal but helpful; avoid funky characters
    desc = f"{label}|txid:{taxid}|bitscore:{stats.get('bitscore','')}|evalue:{stats.get('evalue','')}"
    out_rec = SeqRecord(full.seq, id=acc, description=desc)
    append_fasta_record(MSA_FASTA, out_rec)

    # 4) Summary row incrementally
    row = {
        "label": label,
        "taxid": taxid,
        "ncbi_accession": acc,
        "status": "hit",
        **stats
    }
    append_summary_row(SUMMARY_CSV, row)

    # 5) iTOL datasets incrementally
    display_label = make_display_label(label, acc, taxid)
    popup_text = make_popup_text(label, taxid, acc, stats, stats.get("title",""))
    append_itol_label(acc, display_label)
    append_itol_popup(acc, popup_text)

    # 6) Mark done (so reruns skip this taxon)
    append_line(DONE_TSV, [taxid, label])
    done_taxids.add(taxid)

    polite_sleep()

print("Done. Outputs:")
print("  FASTA   :", MSA_FASTA)
print("  Summary :", SUMMARY_CSV)
print("  iTOL labels:", ITOL_LABELS)
print("  iTOL popup :", ITOL_POPUP)
print("  State dir  :", STATE_DIR)

Already completed taxa: 0
TaxID 696747 (Arthrospira platensis NIES-39) ... 

# Optional: wipe previous NB03 run

In [ ]:
import os, shutil

# DANGER: deletes previous run products from this notebook.
# Change to True ONLY when you deliberately want to erase the run.
WIPE_PREVIOUS_RUN = False

if WIPE_PREVIOUS_RUN:
    old_state = f"{NOTEBOOK_OUTPUT_DIR}/_state"
    to_remove = [
        old_state,
        SUMMARY_CSV,
        MSA_FASTA,
        f"{NOTEBOOK_OUTPUT_DIR}/iTOL_LABELS.tsv",
        f"{NOTEBOOK_OUTPUT_DIR}/iTOL_POPUP.tsv",
    ]

    for p in to_remove:
        if os.path.isdir(p):
            shutil.rmtree(p)
            print("Deleted dir:", p)
        elif os.path.exists(p):
            os.remove(p)
            print("Deleted file:", p)
        else:
            print("Not found:", p)
else:
    print("Previous run was NOT erased. Set WIPE_PREVIOUS_RUN = True to reset.")


In [ ]:
import os
print("STATE_DIR:", STATE_DIR)
print("DONE_TSV exists:", os.path.exists(DONE_TSV), "size:", os.path.getsize(DONE_TSV) if os.path.exists(DONE_TSV) else 0)
print("SUMMARY_CSV exists:", os.path.exists(SUMMARY_CSV), "size:", os.path.getsize(SUMMARY_CSV) if os.path.exists(SUMMARY_CSV) else 0)
print("MSA_FASTA exists:", os.path.exists(MSA_FASTA), "size:", os.path.getsize(MSA_FASTA) if os.path.exists(MSA_FASTA) else 0)